Importing Libraries

In [76]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import NearestNeighbors
from difflib import get_close_matches

EDA

In [77]:
df=pd.read_csv("dataset.csv")

In [78]:
df.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


In [79]:
df.shape


(10000, 9)

In [80]:
df.isnull().sum()

id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

Handling Missing Values

In [81]:
df["overview"]=df["overview"].fillna('')
df["genre"]=df["genre"].fillna('')

df.isnull().sum()


id                   0
title                0
genre                0
original_language    0
overview             0
popularity           0
release_date         0
vote_average         0
vote_count           0
dtype: int64

Feature Selection

In [82]:
df["tags"]= df["genre"] + " : " + df["overview"]
df.head()


,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count,tags
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862,"Drama,Crime : Framed in the 1940s for the doub..."
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731,"Comedy,Drama,Romance : Raj is a rich, carefree..."
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280,"Drama,Crime : Spanning the years 1945 to 1955,..."
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959,"Drama,History,War : The true story of how busi..."
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811,"Drama,Crime : In the continuing saga of the Co..."


In [83]:
movies=df[['id', 'title', 'genre', 'overview', 'tags']].copy()
movies.drop(columns=["genre","overview"])

,id,title,tags
0,278,The Shawshank Redemption,"Drama,Crime : Framed in the 1940s for the doub..."
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance : Raj is a rich, carefree..."
2,238,The Godfather,"Drama,Crime : Spanning the years 1945 to 1955,..."
3,424,Schindler's List,"Drama,History,War : The true story of how busi..."
4,240,The Godfather: Part II,"Drama,Crime : In the continuing saga of the Co..."
...,...,...,...
9995,10196,The Last Airbender,"Action,Adventure,Fantasy : The story follows t..."
9996,331446,Sharknado 3: Oh Hell No!,"Action,TV Movie,Science Fiction,Comedy,Adventu..."
9997,13995,Captain America,"Action,Science Fiction,War : During World War ..."
9998,2312,In the Name of the King: A Dungeon Siege Tale,"Adventure,Fantasy,Action,Drama : A man named F..."


In [84]:
movies.head()

,id,title,genre,overview,tags
0,278,The Shawshank Redemption,"Drama,Crime",Framed in the 1940s for the double murder of h...,"Drama,Crime : Framed in the 1940s for the doub..."
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance : Raj is a rich, carefree..."
2,238,The Godfather,"Drama,Crime","Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime : Spanning the years 1945 to 1955,..."
3,424,Schindler's List,"Drama,History,War",The true story of how businessman Oskar Schind...,"Drama,History,War : The true story of how busi..."
4,240,The Godfather: Part II,"Drama,Crime",In the continuing saga of the Corleone crime f...,"Drama,Crime : In the continuing saga of the Co..."


Text data to feature vectors

In [85]:
cv = CountVectorizer(max_features=10000, stop_words='english')
vector = cv.fit_transform(movies['tags'].values.astype('U')).toarray()

In [86]:
print(vector)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


KNN

In [87]:
knn = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')
knn.fit(vector)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


Movie Recommendation System

In [89]:
def recommend(movie):
    try:
        movie_index = movies[movies['title'] == movie].index[0]
        movie_vector = vector[movie_index].reshape(1, -1)
        distances, indices = knn.kneighbors(movie_vector)
        distances = distances.flatten()
        indices = indices.flatten()
        print(f"Recommendations for '{movie}':")
        for i in range(1, len(indices)):
            recommended_movie_title = movies.iloc[indices[i]].title
            distance_score = distances[i]
            print(f"- {recommended_movie_title}")
    except IndexError:
            print(f"Movie '{movie}' not found in the dataset.")
movie_name = input("Enter The Name Of The Movie: ")
movie_list = movies['title'].tolist()
match = get_close_matches(movie_name, movie_list, n=1, cutoff=0.6)
if match:
    print(f"\nDid you mean: {match[0]}?")
    recommend(match[0])
else:
    print("Movie not found in the dataset.")


Enter The Name Of The Movie:  godfather



Did you mean: The Godfather?
Recommendations for 'The Godfather':
- The Godfather: Part II
- Blood Ties
- Joker
- Bomb City
- Gotti
- Felon
- Rope
- Batman: The Killing Joke
- The Big Heat
- The Outsider
